In [10]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2026, 1, 7, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 1, 7, 23, 59))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
# df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 65.56it/s]


In [9]:
# from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 
# USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path)

from SDRUtils.products.usd.usd_swaptions import USD_Swaptions, straddle_pricer_from_row
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True)
# sdf.head(50)

Classifying Trades: 100%|██████████| 530/530 [00:00<00:00, 1718.30trade/s]


In [17]:
# sdf[sdf["package_type"] == "RISK_REVERSAL"]

sdf["trade_label"].value_counts()
# sdf["package_type"].value_counts()
# sdf[(sdf["package_type"] == "STRADDLE") & ((sdf["forward_label"] == "3M")) & ((sdf["tenor_label"] == "10Y"))]

trade_label
USD-SOFR-OIS Compound 1D CONSTANT 6Mx30Y RECEIVER EURO VANILLA PHYS                                                                                                                                                                                                                11
USD-SOFR-OIS Compound 1Y CONSTANT 4Mx10Y PAYER EURO VANILLA PHYS                                                                                                                                                                                                                    9
USD-SOFR-OIS Compound 1D CONSTANT 3Mx10Y RECEIVER EURO VANILLA PHYS                                                                                                                                                                                                                 7
USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER EURO VANILLA PHYS / USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y RECEIVER EURO VANILLA PHYS                    

In [12]:
sdf[sdf["trade_label"].str.contains("3Y8M")]

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,upi_underlier_name,unique_product_identifier,platform_identifier,cleared,package_indicator,package_transaction_price,option_premium_amount,package_confidence,package_reason,package_legs_count
126,MODI-TRAD,1654197853000000801,2026-01-07 17:52:22+00:00,2026-01-07,2029-08-29,SWAPTION_RECEIVER,USD-SOFR-COMPOUND 1D CONSTANT 3Y8Mx10Y RECEIVE...,100000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZXZSN00ZVCG,BILT,N,False,,"6,495,000",NaN,None,None
134,MODI-TRAD,1654330552000000701,2026-01-07 18:10:19+00:00,2026-01-07,2029-08-29,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D CONSTANT 3Y8Mx10Y PAYER E...,100000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZXSN072GFF3,BILT,N,False,,"6,495,000",NaN,None,None
320,NEWT-TRAD,1654903290000000401 / 1654903291000000501,2026-01-07 20:00:33+00:00,2026-01-07,2029-08-29,SWAPTION_RECEIVER / SWAPTION_PAYER,USD-SOFR-COMPOUND 1D CONSTANT 3Y8Mx10Y RECEIVE...,150000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZXZSN00ZVCG / QZXSN072GFF3,BILT,N,False,NaN,"10,417,500",1.0,platform=BILT; time_delta_max=0.0s; premium_mo...,2


In [13]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))
pricer

QLIRSwapCurve(_ql_curve_id='USD-SOFR-1D', _ql_curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x000001C758AE3B70> >, _ql_curve_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x000001C758AE3F90> >, _meta_data={'timestamp': datetime.datetime(2026, 1, 7, 0, 0)})

In [16]:
# sdf.loc[285], sdf.loc[286]
# .iloc[0].to_dict()
# df[df["Original Dissemination Identifier"] == 1653435994000001301]
straddle_pricer_from_row(sdf.loc[320], pricer)

(<QuantLib.QuantLib.Swaption; proxy of <Swig Object of type 'ext::shared_ptr< Swaption > *' at 0x000001C758922DF0> >,
 55.82433651207474)

In [13]:
ids = [
    "1656268005000000101",
	"1655894585000000101",
	"1655795163000000301",
	"1655795162000000201",
	"1656192606000000601",
	"1655795161000000101",
]

df[df["Dissemination Identifier"].isin(ids)].to_dict(orient="records")

[{'Dissemination Identifier': '1656268005000000101',
  'Original Dissemination Identifier': '1655795162000000201',
  'Action type': 'MODI',
  'Event type': 'TRAD',
  'Event timestamp': Timestamp('2026-01-07 21:51:19+0000', tz='UTC'),
  'Amendment indicator': False,
  'Asset Class': 'IR',
  'Product name': None,
  'Cleared': 'N',
  'Mandatory clearing indicator': False,
  'Execution Timestamp': Timestamp('2026-01-07 21:51:19+0000', tz='UTC'),
  'Effective Date': Timestamp('2026-01-07 00:00:00'),
  'Expiration Date': Timestamp('2029-01-08 00:00:00'),
  'Maturity date of the underlier': datetime.date(2039, 1, 10),
  'Non-standardized term indicator': False,
  'Platform identifier': 'ISWV',
  'Prime brokerage transaction indicator': False,
  'Block trade election indicator': False,
  'Large notional off-facility swap election indicator': None,
  'Notional amount-Leg 1': '100,000,000',
  'Notional amount-Leg 2': '100,000,000',
  'Notional currency-Leg 1': 'USD',
  'Notional currency-Leg 2':